1.  2060636 - კომერციული ბანკების კაპიტალის ადეკვატურობის მოთხოვნების შესახებ დებულების დამტკიცების თაობაზე   (13)
2.  5679194 - ფინანსური ინსტრუმენტების რისკის კატეგორიებისა და მოსალოდნელი საკრედიტო ზარალის განსაზღვრის წესის დამტკიცების თაობაზე (1 შესწორება)
3.  32962 - კომერციული ბანკების საქმიანობის შესახებ (53)
4.  4964423 - მონაცემებზე დაფუძნებული სტატისტიკური, ხელოვნური ინტელექტის და მანქანური სწავლების მოდელების რისკების მართვის დებულების დამტკიცების თაობაზე (2)
5.  4822603 - ფიზიკური პირის დაკრედიტების შესახებ დებულების დამტკიცების თაობაზე  (7)
 
<br>
7.  4709356 - კომერციულ ბანკებში რისკის პოზიციების კონცენტრაციისა და მსხვილი რისკების შესახებ დებულების დამტკიცების თაობაზე
3.  5945436 - კომერციული ბანკების მიერ აქტივების კლასიფიკაციისა და შესაძლო დანაკარგების რეზერვების შექმნისა და გამოყენების წესის დამტკიცების შესახებ


In [7]:
import requests
from pathlib import Path

BASE_URL = "https://matsne.gov.ge"
DOCUMENT_ID = "4822603"

OUTPUT_DIR = Path("matsne_data/graph_docs/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

url = f"{BASE_URL}/ka/document/view/{DOCUMENT_ID}"

response = requests.get(url, timeout=30)
response.raise_for_status()

output_file = OUTPUT_DIR / f"{DOCUMENT_ID}.html"
output_file.write_text(response.text, encoding="utf-8")

print("Status:", response.status_code)
print("URL:", response.url)
print("Saved:", output_file)

Status: 200
URL: https://matsne.gov.ge/ka/document/view/4822603
Saved: matsne_data/graph_docs/raw/4822603.html


In [9]:
TARGET_DOCUMENTS = {
    "2060636": "კომერციული ბანკების კაპიტალის ადეკვატურობის მოთხოვნების შესახებ დებულების დამტკიცების თაობაზ",
    "5679194": "ფინანსური ინსტრუმენტების რისკის კატეგორიებისა და მოსალოდნელი საკრედიტო ზარალის განსაზღვრის წესის დამტკიცების თაობაზე",
    "32962": "კომერციული ბანკების საქმიანობის შესახებ",
    "4964423": "მონაცემებზე დაფუძნებული სტატისტიკური, ხელოვნური ინტელექტის და მანქანური სწავლების მოდელების რისკების მართვის დებულების დამტკიცების თაობაზე",
    "4822603": "ფიზიკური პირის დაკრედიტების შესახებ დებულების დამტკიცების თაობაზე"
}

In [ ]:
import re
import csv
from pathlib import Path
from bs4 import BeautifulSoup


# ============================================================
# CONFIGURATION
# ============================================================

HTML_DIR = Path("../matsne_data/raw_documents")
OUTPUT_FILE = Path("matsne_data/graph_docs/cross_references.csv")

# ============================================================
# HELPERS
# ============================================================

def get_body_content(soup):
    """
    Prefer the main document content to avoid matching
    navigation menus, headers, and footers.
    """

    for selector in [
        "article",
        "main",
        ".document-content",
        ".document-body",
        "#document-content",
    ]:
        element = soup.select_one(selector)

        if element:
            return element

    return soup.body or soup


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


# ============================================================
# SEARCH
# ============================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

html_files = list(HTML_DIR.rglob("*.html"))

results = []

for html_file in html_files:

    print(f"Scanning: {html_file}")

    try:
        html = html_file.read_text(encoding="utf-8")
        soup = BeautifulSoup(html, "html.parser")
    except Exception as error:
        print(f"Could not read {html_file}: {error}")
        continue

    # Identify the source document, if its ID is in the filename.
    source_id = html_file.stem

    content = get_body_content(soup)
    body_text = normalize_text(content.get_text(" ", strip=True))
    body_lower = body_text.casefold()

    # Inspect links inside the body as well.
    links = [
        (a.get("href", ""), normalize_text(a.get_text(" ", strip=True)))
        for a in content.find_all("a", href=True)
    ]

    for target_id, target_title in TARGET_DOCUMENTS.items():

        # Don't report a document as referencing itself.
        if source_id == target_id:
            continue

        # Match the ID as a standalone number.
        id_pattern = re.compile(
            rf"(?<!\d){re.escape(target_id)}(?!\d)"
        )

        id_in_text = bool(id_pattern.search(body_text))

        id_in_link = any(
            id_pattern.search(href)
            or f"/document/view/{target_id}" in href
            for href, _ in links
        )

        # Match the complete title, if supplied.
        title_in_text = (
            bool(target_title)
            and target_title.casefold() in body_lower
        )

        if id_in_text or id_in_link or title_in_text:

            match_types = []

            if id_in_text:
                match_types.append("ID in body text")

            if id_in_link:
                match_types.append("Link to target document")

            if title_in_text:
                match_types.append("Title in body text")

            results.append({
                "source_file": str(html_file),
                "source_document_id": source_id,
                "target_document_id": target_id,
                "target_title": target_title,
                "match_type": "; ".join(match_types),
            })


# ============================================================
# SAVE RESULTS
# ============================================================

fieldnames = [
    "source_file",
    "source_document_id",
    "target_document_id",
    "target_title",
    "match_type",
]

with OUTPUT_FILE.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print("\nFinished.")
print(f"HTML files scanned: {len(html_files)}")
print(f"References found: {len(results)}")
print(f"Results saved to: {OUTPUT_FILE}")